In [47]:
import pandas as pd
import plotly.express as px

df1 = pd.read_csv("D:/Python/libraries/orders_raw.csv")
df2 = pd.read_csv("D:/Python/libraries/products_info_raw.csv")


df1["Product_ID"] = df1["Product_ID"].replace({"prod_01": "PROD-01"})
df2["Kat"] = df2["Kat"].replace({"periferija": "Periferija"}).astype("category")

df_final = pd.merge(df1, df2, left_on="Product_ID", right_on="PID", how="inner") # left_on i right_on jer kolone nisu istog naziva
df_final.drop(columns=["PID"], inplace=True)

df_final["Neto_Profit"] = ((df_final["Prodajna"] * (1 - df_final["Popust_%"])) * df_final["Kolicina"]).astype(int)
df_final["Datum_Prodaje"] = pd.to_datetime(df_final["Datum_Prodaje"], errors="coerce")

# Kolona "Mesec"

In [48]:
df_final['Mesec'] = df_final['Datum_Prodaje'].dt.strftime('%m - %B')

# Pivot Tabela

In [49]:
pivot_profit = pd.pivot_table(df_final,
                              index="Naziv",
                              columns="Mesec",
                              values="Neto_Profit",
                              aggfunc="sum").fillna(0)

# Vizualizacija Heatmap

In [50]:
fig = px.imshow(pivot_profit,
                labels=dict(x="Mesec", y="Proizvod", color="Profit"),
                x=pivot_profit.columns,
                y=pivot_profit.index,
                color_continuous_scale="RdYlGn",
                aspect="auto",
                template="plotly_dark")

fig.update_layout(title="Diagnostic of Profit: Products vs Months")
fig.show()